# Notebook 02 - Wardrop equilibrium

This notebook constructs and solves a finite-route user-equilibrium problem on the graph produced by Notebook 01.

We model the road network as a directed graph $G=(V,E)$. For each edge $e\in E$, let $x_e\ge 0$ be edge flow and let $t_e(x_e)$ be travel time. For each origin-destination pair $w$, let $\mathcal P_w$ be the finite set of candidate routes considered in this notebook, and let $f_p^w\ge 0$ be the flow assigned to route $p\in\mathcal P_w$.

The demand constraints are

$$\sum_{p\in\mathcal P_w} f_p^w=d_w.$$

Edge flows are induced by route flows through

$$x_e=\sum_w\sum_{p\in\mathcal P_w}\delta_{ep}f_p^w,$$

where $\delta_{ep}=1$ if edge $e$ belongs to route $p$, and $0$ otherwise.

The Wardrop equilibrium is computed through the Beckmann formulation

$$\min_f \sum_{e\in E}\int_0^{x_e(f)}t_e(u)\,du,$$

subject to the demand constraints and nonnegativity of route flows.

The edge travel-time model is the BPR function

$$t_e(x_e)=t_e^0\left(1+0.15\left(\frac{x_e}{c_e}\right)^4\right),$$

where $t_e^0>0$ is free-flow travel time and $c_e>0$ is capacity.

Assumptions:
- O-D demands are synthetic.
- Only a small finite set of candidate routes is used for each O-D pair.
- Capacities are heuristic values derived from road class.
- BPR parameters are not calibrated to measured traffic data.


In [ ]:
import sys
sys.path.insert(0, '..')

import osmnx as ox
import networkx as nx
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import islice

from src.graph_utils import load_graph, bpr

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'CVXPY version: {cp.__version__}')

## 1. Load the enriched graph

The graph is loaded from the GraphML file produced by Notebook 01. Each edge is expected to have length, free-flow travel time, and capacity attributes.


In [ ]:
G = load_graph(RAW_DIR / 'chapinero_drive_enriched.graphml')
for u, v, k, data in G.edges(data=True, keys=True):
    data['t0'] = float(data.get('t0', data.get('length', 60) / 8.33))
    data['capacity'] = float(data.get('capacity', 600))

# Extraer listas de nodes y edges (orden fijo)
nodes_list = list(G.nodes())
edges_list  = list(G.edges(keys=True))   # (u, v, key)

N = len(nodes_list)
E = len(edges_list)

node_idx = {n: i for i, n in enumerate(nodes_list)}
edge_idx = {e: i for i, e in enumerate(edges_list)}

print(f'Nodes: {N:,}  |  Edges: {E:,}')

In [ ]:
edges_list = list(G.edges(keys=True))
t0  = np.array([float(G[u][v][k].get('t0',  G[u][v][k]['length'] / 8.33)) for u, v, k in edges_list])
cap = np.array([float(G[u][v][k].get('capacity', 600)) for u, v, k in edges_list])


print(f't0   — min: {t0.min():.1f}s  max: {t0.max():.1f}s  mean: {t0.mean():.1f}s')
print(f'cap  — min: {cap.min():.0f}  max: {cap.max():.0f}  mean: {cap.mean():.0f} veh/h')

## 2. Define a small O-D instance

The O-D pairs are selected heuristically from high-degree nodes. The demands are synthetic and are used only to define a numerical instance.


In [ ]:
# Select the 20 nodes with highest out-degree as O-D candidates
degree_dict = dict(G.out_degree())
top_nodes = sorted(degree_dict, key=degree_dict.get, reverse=True)[:20]

# Create 10 O-D pairs with synthetic demand (veh/h)
np.random.seed(42)
od_pairs = []
for i in range(0, min(20, len(top_nodes)), 2):
    o, d = top_nodes[i], top_nodes[i+1]
    if o != d and nx.has_path(G, o, d):
        demand = np.random.randint(100, 400)  # veh/h
        od_pairs.append((o, d, demand))

print(f'O-D pairs definidos: {len(od_pairs)}')
for o, d, dem in od_pairs:
    print(f'  {o} → {d}  |  demand = {dem} veh/h')

## 3. Enumerate candidate routes

For each O-D pair, the notebook keeps at most $k=3$ shortest simple paths with respect to free-flow travel time. The equilibrium is therefore computed over this restricted route set.


In [ ]:
def k_shortest_paths(G, source, target, k, weight='t0'):
    """Genera hasta k shortest paths entre source y target."""
    try:
        gen = nx.shortest_simple_paths(G, source, target, weight=weight)
        return list(islice(gen, k))
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return []

K = 3  # routes por par O-D
routes = {}   # (o, d) -> lista de routes (cada route = lista de nodes)
G_simple = nx.DiGraph(G)

for o, d, dem in od_pairs:
    paths = k_shortest_paths(G_simple, o, d, K, weight='t0')
    if paths:
        routes[(o, d)] = paths

total_routes = sum(len(v) for v in routes.values())
print(f'Routes enumeradas: {total_routes} ({K} por par O-D)')

## 4. Build the route-edge incidence matrix

The incidence coefficient is

$$\delta_{ep}=\begin{cases}1,& e\in p,\\0,& e\notin p.\end{cases}$$

The incidence matrix maps route flows to edge flows.


In [ ]:
# Global route indices
route_list = []  # (od_pair_index, route)
od_route_map = {}  # od_pair -> [route_global_indices]

for od, paths in routes.items():
    od_route_map[od] = []
    for path in paths:
        od_route_map[od].append(len(route_list))
        route_list.append((od, path))

R = len(route_list)

# Delta matrix: E x R (sparse as a dict before construction)
# Para CVXPY usamos enfoque de flow en edges directamente
print(f'Total de routes: {R}')
print(f'Delta size: {E} edges × {R} routes')

## 5. Beckmann objective in CVXPY

For BPR costs with exponent $4$, the edge contribution to the Beckmann objective is

$$\int_0^{x_e}t_e(u)\,du=t_e^0x_e+\frac{0.15\,t_e^0}{5c_e^4}x_e^5.$$

This expression is convex for $x_e\ge 0$ when $t_e^0>0$ and $c_e>0$.


In [ ]:
def beckmann_bpr(x, t0, cap, alpha=0.15, beta=4):
    """
    Beckmann integral with BPR.
    x, t0, cap: vectores de edges (numpy o cp.Variable)
    Return a scalar CVXPY expression.
    """
    # t0 * x  +  t0 * alpha/(beta+1) * x^(beta+1) / cap^beta
    # Para beta=4: t0*x + t0*0.15/5 * x^5 / cap^4
    # cp.power(x, 5) is convex for x>=0
    term_linear = cp.sum(cp.multiply(t0, x))
    term_power  = cp.sum(
        cp.multiply(t0 * alpha / (beta + 1) / (cap ** beta),
                    cp.power(x, beta + 1))
    )
    return term_linear + term_power

In [ ]:
# Variables de flow por route (f_r >= 0)
f = cp.Variable(R, nonneg=True, name='flow_routes')

# Construir matriz delta como numpy array
delta = np.zeros((E, R))
for r_idx, (od, path) in enumerate(route_list):
    for i in range(len(path) - 1):
        u, v = path[i], path[i+1]
        # Encontrar la edge (puede haber multigrafo)
        if G.has_edge(u, v):
            # Take the edge with the smallest t0
            best_k = min(G[u][v], key=lambda k: G[u][v][k].get('t0', float('inf')))
            e_key = (u, v, best_k)
            if e_key in edge_idx:
                delta[edge_idx[e_key], r_idx] = 1

# Flows en edges: x = delta @ f
x = delta @ f   # CVXPY expression, shape (E,)

print(f'Matriz delta construida: {delta.shape}')
print(f'Densidad: {delta.mean()*100:.2f}%')

In [ ]:
# Demand-conservation constraints
constraints = []
for (o, d, dem) in od_pairs:
    od_key = (o, d)
    if od_key in od_route_map:
        route_idxs = od_route_map[od_key]
        constraints.append(cp.sum(f[route_idxs]) == dem)

# Optimization problem (Beckmann UE)
objective = cp.Minimize(beckmann_bpr(x, t0, cap))
problem_ue = cp.Problem(objective, constraints)

print(f'Problema UE:')
print(f'  Variables      : {R} (flows de route)')
print(f'  Constraints  : {len(constraints)}')
print(f'  Is DCP/convex?: {problem_ue.is_dcp()}')

## 6. Solve the finite-dimensional convex problem

The solver is applied to the restricted route set. Solver status and numerical accuracy must be checked before interpreting the output.


In [ ]:
problem_ue.solve(solver=cp.SCS, verbose=False)

print(f'Status        : {problem_ue.status}')
print(f'Optimal value  : {problem_ue.value:.4f}')

if problem_ue.status not in ['optimal', 'optimal_inaccurate']:
    print('WARNING: the solver did not find an optimal solution.')
    print(f'O-D pairs with routes: {len(od_route_map)}')
    print(f'O-D pairs totales  : {len(od_pairs)}')
    print(f'Constraints      : {len(constraints)}')
else:
    x_ue = delta @ f.value
    t_ue = np.array([bpr(t0[i], x_ue[i], cap[i]) for i in range(E)])
    print(f'\nUE flow statistics:')
    print(f'  Edges with flow > 0 : {(x_ue > 1).sum()}')
    print(f'  Maximum flow          : {x_ue.max():.1f} veh/h')
    print(f'  Mean flow (actives) : {x_ue[x_ue>1].mean():.1f} veh/h')

## 7. Compute total system cost at UE

After solving the Beckmann problem, total system cost is evaluated as

$$\mathrm{TC}(x^{UE})=\sum_{e\in E}x_e^{UE}t_e(x_e^{UE}).$$


In [ ]:
# Total cost = suma de (time en edge * flow en edge)
cost_ue = float(np.sum(t_ue * x_ue))
print(f'Total cost UE (veh·s): {cost_ue:,.0f}')
print(f'Total cost UE (veh·h): {cost_ue/3600:,.1f}')

# Vehicle-weighted average time
total_demand = sum(dem for _, _, dem in od_pairs)
avg_time_ue  = cost_ue / total_demand
print(f'Demand total          : {total_demand} veh/h')
print(f'Average time UE     : {avg_time_ue:.1f} s ({avg_time_ue/60:.2f} min)')

## 8. Check the Wardrop condition on candidate routes

For each O-D pair, used candidate routes should have equal minimal route cost up to numerical tolerance. This check is limited to the enumerated route set.


In [ ]:
# Construir diccionario de times de edge en UE
t_ue_dict = {edges_list[i]: t_ue[i] for i in range(E)}

print('Wardrop condition check by O-D pair:')
print(f'{"Par O-D":>30}  {"Routes":>5}  {"Minimum cost":>10}  {"Maximum cost":>10}  {"Gap %":>7}')
print('-' * 70)

for (o, d, dem) in od_pairs:
    od_key = (o, d)
    if od_key not in od_route_map:
        continue
    route_idxs = od_route_map[od_key]
    f_vals = f.value[route_idxs]
    
    route_costs = []
    for r_idx, (_, path) in enumerate(route_list):
        if r_idx in route_idxs:
            cost = sum(
                t_ue_dict.get((path[i], path[i+1], 
                               min(G[path[i]][path[i+1]], 
                                   key=lambda k: G[path[i]][path[i+1]][k].get('t0', float('inf')))), 0)
                for i in range(len(path)-1)
            )
            route_costs.append(cost)
    
    if route_costs:
        c_min, c_max = min(route_costs), max(route_costs)
        gap = (c_max - c_min) / c_min * 100 if c_min > 0 else 0
        print(f'{str(od_key):>30}  {len(route_costs):>5}  {c_min:>10.1f}  {c_max:>10.1f}  {gap:>7.2f}%')

## 9. Visualization

The plots display computed flows and costs for inspection. They should not be interpreted as validation against observed traffic counts.


In [ ]:
# Assign UE flows to the graph for visualization
for i, (u, v, k) in enumerate(edges_list):
    G[u][v][k]['flow_ue']      = float(x_ue[i])
    G[u][v][k]['travel_time_ue'] = float(t_ue[i])

# Colorear por flow
edge_colors = ox.plot.get_edge_colors_by_attr(G, attr='flow_ue', cmap='hot_r', na_color='#2a2a4a')
edge_widths  = [min(4, 0.5 + G[u][v][k]['flow_ue'] / 200) for u, v, k in edges_list]

fig, ax = ox.plot_graph(
    G,
    edge_color=edge_colors,
    edge_linewidth=edge_widths,
    node_size=5,
    node_color='white',
    bgcolor='#1a1a2e',
    figsize=(12, 10),
    show=False,
    close=False,
)
ax.set_title('Wardrop equilibrium - edge flows (veh/h)\nChapinero, Bogota',
             color='white', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'ue_flows.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Flow histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

active = x_ue[x_ue > 1]
axes[0].hist(active, bins=30, color='#e74c3c', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Flujo UE (veh/h)', fontsize=11)
axes[0].set_ylabel('Number of edges', fontsize=11)
axes[0].set_title('Distribution of equilibrium flows')

axes[1].hist(t_ue[x_ue > 1], bins=30, color='#3498db', edgecolor='white', linewidth=0.4)
axes[1].set_xlabel('Tiempo de viaje UE (s)', fontsize=11)
axes[1].set_ylabel('Number of edges', fontsize=11)
axes[1].set_title('Distribution of equilibrium travel times')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'ue_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save intermediate results

The saved object records this numerical instance and the computed UE quantities. Later notebooks depend on this file.

Limitations of this notebook:
- The equilibrium is defined only over the enumerated candidate routes.
- The demand vector is synthetic.
- Solver status and residuals should be checked before using numerical values.


In [ ]:
import pickle

results_ue = {
    'x_ue':          x_ue,           # edge flows (E,)
    't_ue':          t_ue,           # equilibrium travel times (E,)
    't0':            t0,             # times de viaje libres (E,)
    'cap':           cap,            # capacityes (E,)
    'cost_ue':       cost_ue,        # cost total sistema (veh·s)
    'edges_list':    edges_list,     # lista de edges (u, v, k)
    'od_pairs':      od_pairs,       # O-D pairs with demand
    'route_list':    route_list,     # routes enumeradas
    'od_route_map':  od_route_map,   # O-D mapping to route indices
    'delta':         delta,          # matriz incidencia edge-route
}

out_path = PROCESSED_DIR / 'results_ue.pkl'
with open(out_path, 'wb') as fh:
    pickle.dump(results_ue, fh)

print(f'UE results saved to: {out_path}')
print(f'\nResumen:')
print(f'  Total cost UE : {cost_ue:,.0f} veh·s')
print(f'  Tiempo prom.   : {avg_time_ue:.1f} s')
print(f'  → Ready for Notebook 03 — Social optimum')